In [ ]:
!pip install networkx matplotlib torch torchvision scipy pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.ndimage import gaussian_filter
from tqdm import tqdm

os.makedirs("density_dataset/images", exist_ok=True)
os.makedirs("density_dataset/density", exist_ok=True)

IMG_SIZE = 128
NUM_SAMPLES = 10000

def generate_density_map(points, h, w):

    density = np.zeros((h, w), dtype=np.float32)

    for x, y in points:
        if x >= w or y >= h:
            continue
        density[y, x] = 1

    density = gaussian_filter(density, sigma=3)

    return density


for i in tqdm(range(NUM_SAMPLES)):

    num_nodes = random.randint(10, 100)

    G = nx.gnp_random_graph(num_nodes, p=random.uniform(0.1,0.4))

    pos = nx.random_layout(G)

    coords = []

    fig = plt.figure(figsize=(2,2))

    for node in G.nodes():
        x,y = pos[node]

        px = int((x+1)/2 * IMG_SIZE)
        py = int((y+1)/2 * IMG_SIZE)

        coords.append((px,py))

    nx.draw(G, pos, node_size=50, node_color="black", edge_color="gray", with_labels=False)

    plt.axis("off")

    img_path = f"density_dataset/images/{i}.png"
    plt.savefig(img_path, bbox_inches="tight", pad_inches=0)
    plt.close()

    density = generate_density_map(coords, IMG_SIZE, IMG_SIZE)

    np.save(f"density_dataset/density/{i}.npy", density)

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class GraphDensityDataset(Dataset):

    def __init__(self, img_dir, density_dir):

        self.img_dir = img_dir
        self.density_dir = density_dir
        self.files = os.listdir(img_dir)

        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((128,128)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        name = self.files[idx].split(".")[0]

        img = Image.open(f"{self.img_dir}/{name}.png")
        img = self.transform(img)

        density = np.load(f"{self.density_dir}/{name}.npy")

        density = torch.tensor(density).unsqueeze(0)

        original_sum = density.sum()

        density = torch.nn.functional.interpolate(
            density.unsqueeze(0),
            size=(32,32),
            mode='bilinear',
            align_corners=False
        ).squeeze(0)

        # Preserve node count
        density = density * (original_sum / density.sum())

        return img, density

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DensityCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Frontend
        self.frontend = nn.Sequential(

            nn.Conv2d(1,64,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.Conv2d(64,64,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),

            nn.Conv2d(128,128,3,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),

            nn.MaxPool2d(2),
        )

        # Backend (dilated conv)
        self.backend = nn.Sequential(

            nn.Conv2d(128,256,3,padding=2,dilation=2),
            nn.ReLU(),
            nn.BatchNorm2d(256),

            nn.Conv2d(256,128,3,padding=2,dilation=2),
            nn.ReLU(),
            nn.BatchNorm2d(128),

            nn.Conv2d(128,64,3,padding=2,dilation=2),
            nn.ReLU(),
            nn.BatchNorm2d(64)
        )

        self.output_layer = nn.Conv2d(64,1,1)

    def forward(self,x):

        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)

        return x

In [ ]:
from torch.utils.data import DataLoader, random_split

dataset = GraphDensityDataset(
    "density_dataset/images",
    "density_dataset/density"
)

train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset,[train_size,test_size])

train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DensityCNN().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)

In [ ]:
def train():

    model.train()

    total_loss = 0

    for imgs, density in train_loader:

        imgs = imgs.to(device)
        density = density.to(device)

        optimizer.zero_grad()

        pred = model(imgs)

        loss = criterion(pred, density)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Train Loss:", total_loss/len(train_loader))

In [ ]:
def evaluate(loader):

    model.eval()

    mae = 0

    with torch.no_grad():

        for imgs, density in loader:

            imgs = imgs.to(device)
            density = density.to(device)

            pred = model(imgs)

            pred_count = pred.sum(dim=[1,2,3])
            true_count = density.sum(dim=[1,2,3])

            mae += torch.mean(torch.abs(pred_count-true_count)).item()

    print("MAE:", mae/len(loader))

In [ ]:
import pickle
import gzip
def save_model(net, filename):
  with gzip.open(filename, 'wb') as f:
    pickle.dump(net, f)

def load_model(filename):
  with gzip.open(filename, 'rb') as f:
    return pickle.load(f)

In [ ]:
EPOCHS = 1

for epoch in range(EPOCHS):

    print("\nEpoch",epoch+1)

    train()

    evaluate(train_loader)

    evaluate(test_loader)

In [ ]:
save_model(model, '/content/drive/MyDrive/honors_thesis_data/density_random_layout_model.pkl.gz')

In [ ]:
model = load_model('/content/drive/MyDrive/honors_thesis_data/density_random_layout_model.pkl.gz')
model.to(device)

In [ ]:
img, density = test_dataset[0]

model.eval()

with torch.no_grad():

    pred = model(img.unsqueeze(0).to(device))

count = pred.sum().item()

print("Predicted nodes:", round(count))
print("True nodes:", density.sum().item())

In [ ]:
import matplotlib.pyplot as plt

img, density = test_dataset[2]

model.eval()

with torch.no_grad():
    pred = model(img.unsqueeze(1).to(device)).cpu()

pred_density = pred.squeeze().numpy()
true_density = density.squeeze().numpy()
image = img.squeeze().numpy()

pred_count = pred_density.sum()
true_count = true_density.sum()

print("Predicted nodes:", round(pred_count,2))
print("True nodes:", round(true_count,2))


plt.figure(figsize=(12,4))

# Original image
plt.subplot(1,3,1)
plt.imshow(image, cmap="gray")
plt.title("Graph Image")
plt.axis("off")

# True density map
plt.subplot(1,3,2)
plt.imshow(true_density, cmap="jet")
plt.title(f"True Density\nCount={round(true_count,2)}")
plt.colorbar()
plt.axis("off")

# Predicted density map
plt.subplot(1,3,3)
plt.imshow(pred_density, cmap="jet")
plt.title(f"Predicted Density\nCount={round(pred_count,2)}")
plt.colorbar()
plt.axis("off")

plt.show()

In [ ]:
plt.imshow(image, cmap="gray")
plt.imshow(pred_density, cmap="jet", alpha=0.5)
plt.title("Predicted Density Overlay")
plt.axis("off")
plt.show()